# 01.3 - Linked Lists

## What is a linked list?
A chain of **nodes**. Each node stores a value and a **pointer to the next node**. Nodes can be anywhere in memory.

```
head
 |
[1 | •]--> [2 | •]--> [3 | •]--> None        singly linked

None <--[• | 1 | •]<-->[• | 2 | •]<-->[• | 3 | •]--> None   doubly linked
```

## Array vs linked list
| | Array (list) | Linked list |
|---|---|---|
| Access i-th element | **O(1)** | O(n) |
| Insert/delete at front | O(n) | **O(1)** |
| Insert/delete given a node | O(n) | **O(1)** |
| Memory | Compact | Extra pointer per node |

## Where is it used?
| Real system | Use |
|---|---|
| **LRU caches** (Redis, browsers, CPUs) | Doubly linked list + hash map (built below) |
| Music / video playlists | Next / previous song |
| Browser history | Back / forward |
| Operating systems | Process lists, free memory blocks |
| Python's `collections.deque` | Implemented as a doubly linked list of blocks |
| Blockchain | Each block points to the previous block's hash |
| Undo history in editors | Chain of states |

## 1. Build a singly linked list from scratch

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_list(values):
    """[1,2,3] -> 1 -> 2 -> 3  (test helper, not needed on LeetCode)"""
    dummy = cur = ListNode()
    for v in values:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def list_to_py(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

In [2]:
class LinkedList:
    def __init__(self):
        self.head = None
        self.size = 0

    def push_front(self, val):          # O(1)
        self.head = ListNode(val, self.head)
        self.size += 1

    def push_back(self, val):           # O(n) (O(1) if we also keep a tail pointer)
        node = ListNode(val)
        if not self.head:
            self.head = node
        else:
            cur = self.head
            while cur.next:
                cur = cur.next
            cur.next = node
        self.size += 1

    def find(self, val):                # O(n)
        cur = self.head
        while cur:
            if cur.val == val:
                return True
            cur = cur.next
        return False

    def delete(self, val):              # O(n) to find, O(1) to unlink
        dummy = ListNode(0, self.head)
        prev, cur = dummy, self.head
        while cur:
            if cur.val == val:
                prev.next = cur.next
                self.size -= 1
                break
            prev, cur = cur, cur.next
        self.head = dummy.next

    def __iter__(self):
        cur = self.head
        while cur:
            yield cur.val
            cur = cur.next

    def __repr__(self):
        return " -> ".join(map(str, self)) + " -> None"

ll = LinkedList()
for x in [2, 3, 4]:
    ll.push_back(x)
ll.push_front(1)
print(ll)
ll.delete(3)
print(ll, "| size:", ll.size, "| has 4?", ll.find(4))
assert list(ll) == [1, 2, 4]

1 -> 2 -> 3 -> 4 -> None
1 -> 2 -> 4 -> None | size: 3 | has 4? True


## 2. Key techniques
1. **Dummy node** - a fake node before `head` so deleting/inserting at the head isn't a special case.
2. **Pointer reversal** - `prev, curr, nxt`.
3. **Slow & fast pointers** - middle of list, cycle detection.
4. **Draw it!** On paper or in comments - linked-list bugs are pointer bugs.

---
## Solved problems

### 1. Middle of the Linked List - LC #876 (Easy)
Return the middle node (second middle if even length).

**Idea:** Fast moves 2 steps, slow moves 1. When fast reaches the end, slow is in the middle.

In [3]:
def middle_node(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

assert middle_node(build_list([1, 2, 3, 4, 5])).val == 3
assert middle_node(build_list([1, 2, 3, 4, 5, 6])).val == 4
print("Middle Node ")

Middle Node 


### 2. Palindrome Linked List - LC #234 (Easy)
Is the list a palindrome? Do it in O(1) extra space.

**Idea:** Find the middle (slow/fast), reverse the second half, compare both halves.

In [4]:
def is_palindrome_list(head):
    # 1) find middle
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    # 2) reverse second half
    prev = None
    while slow:
        slow.next, prev, slow = prev, slow, slow.next
    # 3) compare halves
    left, right = head, prev
    while right:
        if left.val != right.val:
            return False
        left, right = left.next, right.next
    return True

assert is_palindrome_list(build_list([1, 2, 2, 1]))
assert is_palindrome_list(build_list([1, 2, 3, 2, 1]))
assert not is_palindrome_list(build_list([1, 2]))
print("Palindrome List ")

Palindrome List 


### 3. Intersection of Two Linked Lists - LC #160 (Easy)
Return the node where two lists merge, or None.

**Idea:** Walk A then B, and B then A. Both pointers travel the same total distance, so they meet at the intersection (or both reach None).

In [5]:
def get_intersection_node(a, b):
    p, q = a, b
    while p is not q:
        p = p.next if p else b      # switch to the other list at the end
        q = q.next if q else a
    return p

common = build_list([8, 4, 5])
a = build_list([4, 1]); a.next.next = common
b = build_list([5, 6, 1]); b.next.next.next = common
assert get_intersection_node(a, b) is common
assert get_intersection_node(build_list([1]), build_list([2])) is None
print("Intersection ")

Intersection 


### 4. Add Two Numbers - LC #2 (Medium) Digits stored in reverse order (342 = 2→4→3). Return the sum as a list.

**Idea:** Grade-school addition digit by digit with a carry; dummy node to build the result.

In [6]:
def add_two_numbers(l1, l2):
    dummy = cur = ListNode()
    carry = 0
    while l1 or l2 or carry:
        total = carry
        if l1:
            total += l1.val; l1 = l1.next
        if l2:
            total += l2.val; l2 = l2.next
        carry, digit = divmod(total, 10)
        cur.next = ListNode(digit)
        cur = cur.next
    return dummy.next

assert list_to_py(add_two_numbers(build_list([2, 4, 3]), build_list([5, 6, 4]))) == [7, 0, 8]   # 342+465=807
assert list_to_py(add_two_numbers(build_list([9, 9, 9]), build_list([1]))) == [0, 0, 0, 1]
print("Add Two Numbers ")

Add Two Numbers 


### 5. LRU Cache - LC #146 (Medium) real-world system design
Design a cache with capacity `c`: `get(key)` and `put(key, value)` in **O(1)**. When full, evict the **Least Recently Used** key.
This is exactly how Redis, browsers and CPU caches decide what to throw away.

**Idea:** Hash map gives O(1) lookup of the node; a **doubly linked list** keeps usage order with O(1) move-to-front and remove-from-back.

In [7]:
class DNode:
    def __init__(self, key=0, val=0):
        self.key, self.val = key, val
        self.prev = self.next = None

class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.map = {}                         # key -> node
        self.head, self.tail = DNode(), DNode()   # dummies: head <-> ... <-> tail
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node):
        node.prev.next, node.next.prev = node.next, node.prev

    def _add_front(self, node):               # most recently used goes to the front
        node.next, node.prev = self.head.next, self.head
        self.head.next.prev = node
        self.head.next = node

    def get(self, key):
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node)
        self._add_front(node)
        return node.val

    def put(self, key, value):
        if key in self.map:
            self._remove(self.map[key])
        node = DNode(key, value)
        self.map[key] = node
        self._add_front(node)
        if len(self.map) > self.capacity:
            lru = self.tail.prev               # least recently used
            self._remove(lru)
            del self.map[lru.key]

cache = LRUCache(2)
cache.put(1, 1); cache.put(2, 2)
assert cache.get(1) == 1          # 1 is now most recent
cache.put(3, 3)                   # evicts 2
assert cache.get(2) == -1
cache.put(4, 4)                   # evicts 1
assert cache.get(1) == -1 and cache.get(3) == 3 and cache.get(4) == 4
print("LRU Cache ")

# Python shortcut (fine to mention in interviews): collections.OrderedDict has move_to_end + popitem(last=False)

LRU Cache 


## More practice on LeetCode
Easy: Remove Linked List Elements (#203), Remove Duplicates from Sorted List (#83)
Medium: Reorder List (#143), Copy List with Random Pointer (#138), Linked List Cycle II (#142), Sort List (#148)
Hard: Merge k Sorted Lists (#23 - solved in 01.6), Reverse Nodes in k-Group (#25)

Also in `03-Interview Patterns/03.1`: Reverse List, Merge Two Lists, Cycle Detection, Remove Nth From End.

## Key takeaways
- O(1) insert/delete when you have the node; O(n) to find by index.
- Use a **dummy node**, **slow/fast pointers**, and **draw the pointers**.
- Hash map + doubly linked list = LRU cache (a classic system-design question).

**Next: `01.4-Stacks-and-Queues.ipynb`**